<a href="https://colab.research.google.com/github/ebuillent13yadav/Sign_Detection/blob/main/prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import cv2
from tensorflow.keras.models import load_model

In [4]:
model = load_model("gesture_model.keras")
img_size = 128
class_names = ['APPROVE', 'CALL_ME', 'DISLIKE', 'DOMAIN_EXPANSION_INFINITE_VOID', 'HELP', "I_LOVE_YOU", 'LOOK', 'OK', 'PEACE', 'STOP']

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
def predict_frame(frame):
  img = cv2.resize(frame, (img_size, img_size))
  img = img / 255.0
  img = np.expand_dims(img, axis = 0)

  predictions = model.predict(img, verbose=0)
  confidence = np.max(predictions)
  class_index = np.argmax(predictions)
  gesture = class_names[class_index]

  return gesture, confidence

In [6]:
cap = cv2.VideoCapture(0)

while True:
  ret, frame = cap.read()
  frame = cv2.flip(frame, 1)

  h, w, _ = frame.shape

  # ROI
  box_size = 300
  x1 = w//2 - box_size//2
  y1 = h//2 - box_size//2
  x2 = x1 + box_size
  y2 = y1 + box_size

  cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

  roi = frame[y1:y2, x1:x2]

  gesture, confidence = predict_frame(roi)

  if confidence > 0.6:
    cv2.putText(frame, f"{gesture} ({confidence})",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,0), 2)

    cv2.imshow("Gesture Recognition", frame)

    if cv2.waitkey(1) & 0xFF == 27:
      break

cap.release()
cv2.destroyAllWindows()

AttributeError: 'NoneType' object has no attribute 'shape'